# 修改、刷新和新增实验

这是独立的合成沙盒，不连接设备。无需完成其他课程。先从新 kernel 顺序运行，再做文末的小修改。
启动入口已准备环境与服务；重复打开继续当前练习，重置会创建新的起点。

In [ ]:
import scopecat as sc

session = sc.notebook()
session

In [ ]:
from my_experiment.setup import open_parameters
from my_experiment.teaching import teaching_rabi

params = open_parameters(session)

下面先保留一个请求。用 VS Code 修改 `src/my_experiment/teaching.py` 的 seed 默认值，保存后直接执行后面的新请求单元；无需 refresh 或重新 import。旧请求仍保持原版本。

In [ ]:
before = teaching_rabi()
print("旧默认值:", before.values["seed"])

In [ ]:
after = teaching_rabi()
print("旧请求、新请求:", before.values["seed"], after.values["seed"])
prepared = session.prepare(after.sweep(amplitude=[0.1, 0.2]), parameters=params)
assert prepared.preview.point_count == 2
run = prepared.run().wait(timeout=120).result()
print("run:", run.id)

新增实验：已有可编辑起点 `examples/extra.py`。下面将它复制进作者源码目录。重新执行该单元不会覆盖你已经修改的文件；也可以用 VS Code 手动复制同一个文件。

In [ ]:
from shutil import copyfile

project = sc.open_project()
extra = project.root / "src/my_experiment/extra.py"
if not extra.exists():
    copyfile(project.root / "examples/extra.py", extra)

In [ ]:
import my_experiment.extra as extra_experiments

new_request = extra_experiments.extra_rabi().sweep(amplitude=[0.1, 0.2])
new_run = (
    session.prepare(new_request, parameters=params).run().wait(timeout=120).result()
)
assert len(new_run.measurements()) == 2
print("新实验:", new_request.declaration.id, "run:", session.run_number(new_run))

小修改：改变新实验的默认 seed，保存后直接重跑上一格。已有实验别名会解析最新定义，新模块在下一格开始前被发现。修改参数对象仍需自行保存；普通 Python 函数和实例不会被任意改写。

若制造语法错误，观察当前输出中的错误，修复文件后重试；不用重新安装。需要暂停自动更新时执行 `sc.notebook(live=False)`，恢复时用 `sc.notebook(live=True)`。

重启 kernel 后，只执行初始化单元，然后用 `session.history()` 找回目标编号、用 `session.run(编号)` 读取，不必重新运行实验。重复初始化会复用同一个 session；不同 Notebook 若共享 kernel，也共享默认工作区。